# Real ONT Training Run

This notebook performs a full end-to-end run on a real Oxford Nanopore dataset for *E. coli* UTI89:

1. verify or download the raw reads and reference,
2. align a notebook-sized subset of real long reads with `mappy`,
3. build overlap-conditioned JSONL windows for OMEGA,
4. train the model for one epoch, and
5. evaluate on a held-out test split.

The run is intentionally sized to complete in a notebook while still using real sequencing data.


In [82]:
from __future__ import annotations

import copy
import gzip
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

import mappy as mp
import pandas as pd
import pysam
import yaml

ROOT = Path('/Users/shanejayasundera/LRS-Error-Correction')
RAW_DIR = ROOT / 'data' / 'raw' / 'uti89'
RUN_DIR = ROOT / 'data' / 'real_uti89_notebook'
CKPT_DIR = ROOT / 'checkpoints' / 'real_uti89_notebook'
OUT_DIR = ROOT / 'outputs' / 'real_uti89_notebook'
NOTEBOOK_CONFIG = ROOT / 'configs' / 'real_uti89_notebook.yaml'

RAW_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_GZ = RAW_DIR / 'reference.fna.gz'
REFERENCE_FA = RAW_DIR / 'reference.fna'
FASTQ_1 = RAW_DIR / 'ERR908493_1.fastq.gz'
FASTQ_2 = RAW_DIR / 'ERR908493_2.fastq.gz'
SORTED_BAM = RUN_DIR / 'aligned.sorted.bam'
MANIFEST = RUN_DIR / 'manifest.json'

print('python', sys.executable)
print('cwd', ROOT)
print('raw dir', RAW_DIR)


python /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python
cwd /Users/shanejayasundera/LRS-Error-Correction
raw dir /Users/shanejayasundera/LRS-Error-Correction/data/raw/uti89


In [83]:
DOWNLOADS = {
    REFERENCE_GZ: 'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/013/265/GCA_000013265.1_ASM1326v1/GCA_000013265.1_ASM1326v1_genomic.fna.gz',
    FASTQ_1: 'https://ftp.sra.ebi.ac.uk/vol1/fastq/ERR908/ERR908493/ERR908493_1.fastq.gz',
    FASTQ_2: 'https://ftp.sra.ebi.ac.uk/vol1/fastq/ERR908/ERR908493/ERR908493_2.fastq.gz',
}

for path, url in DOWNLOADS.items():
    if path.exists() and path.stat().st_size > 0:
        print('exists', path.name, path.stat().st_size)
    else:
        print('downloading', path.name)
        urlretrieve(url, path)
        print('downloaded', path.name, path.stat().st_size)

if (not REFERENCE_FA.exists()) or REFERENCE_FA.stat().st_size == 0:
    with gzip.open(REFERENCE_GZ, 'rb') as src, open(REFERENCE_FA, 'wb') as dst:
        dst.write(src.read())
    print('wrote', REFERENCE_FA)
else:
    print('exists', REFERENCE_FA)

if not (REFERENCE_FA.with_suffix(REFERENCE_FA.suffix + '.fai')).exists():
    pysam.faidx(str(REFERENCE_FA))
    print('indexed reference')
else:
    print('reference index exists')


exists reference.fna.gz 1540635
exists ERR908493_1.fastq.gz 9018804
exists ERR908493_2.fastq.gz 10102609
exists /Users/shanejayasundera/LRS-Error-Correction/data/raw/uti89/reference.fna
reference index exists


In [84]:
sys.path.insert(0, str(ROOT / 'src'))

import importlib
import inspect
import omega_longread.preprocessing as preprocessing
preprocessing = importlib.reload(preprocessing)
VariantLookup = preprocessing.VariantLookup
build_read_encoding = preprocessing.build_read_encoding
build_window_example = preprocessing.build_window_example
iter_windows = preprocessing.iter_windows

MAX_READS = 1000
MIN_READ_LENGTH = 1500
WINDOW_SIZE = 1024
WINDOW_OVERLAP = 256
MAX_SUPPORTS = 8
MIN_SUPPORTS_PER_WINDOW = 1
MIN_MAPQ = 5
MAX_INSERTIONS_PER_POS = 2

UNSORTED_BAM = RUN_DIR / 'aligned.unsorted.bam'
TRAIN_JSONL = RUN_DIR / 'train.jsonl'
VAL_JSONL = RUN_DIR / 'val.jsonl'
TEST_JSONL = RUN_DIR / 'test.jsonl'


def revcomp(seq: str) -> str:
    table = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
    return seq.translate(table)[::-1]


def iter_fastq(path: Path):
    with gzip.open(path, 'rt') as f:
        while True:
            name = f.readline().strip()
            if not name:
                return
            seq = f.readline().strip()
            f.readline()
            qual = f.readline().strip()
            yield name[1:], seq, qual


def full_cigar(hit, query_len: int) -> str:
    if hit.strand < 0:
        left_soft = query_len - hit.q_en
        right_soft = hit.q_st
    else:
        left_soft = hit.q_st
        right_soft = query_len - hit.q_en
    parts = []
    if left_soft > 0:
        parts.append(f'{left_soft}S')
    parts.append(hit.cigar_str)
    if right_soft > 0:
        parts.append(f'{right_soft}S')
    return ''.join(parts)

reads = []
for fq in [FASTQ_1, FASTQ_2]:
    for name, seq, qual in iter_fastq(fq):
        if len(seq) >= MIN_READ_LENGTH:
            reads.append((name, seq, qual, fq.name))
reads.sort(key=lambda x: len(x[1]), reverse=True)
reads = reads[:MAX_READS]
print('selected reads', len(reads))

aligner = mp.Aligner(str(REFERENCE_FA), preset='map-ont')
fasta = pysam.FastaFile(str(REFERENCE_FA))
header = {
    'HD': {'VN': '1.6', 'SO': 'unsorted'},
    'SQ': [{'SN': name, 'LN': length} for name, length in zip(fasta.references, fasta.lengths)],
}
ref_to_id = {name: idx for idx, name in enumerate(fasta.references)}

written = 0
with pysam.AlignmentFile(str(UNSORTED_BAM), 'wb', header=header) as bam:
    for name, seq, qual, source in reads:
        hits = [h for h in aligner.map(seq) if h.is_primary]
        if not hits:
            continue
        hit = max(hits, key=lambda h: (h.mapq, h.mlen, h.blen))
        aln = pysam.AlignedSegment()
        aln.query_name = f'{source}|{name}'
        aln.flag = 16 if hit.strand < 0 else 0
        aln.reference_id = ref_to_id[hit.ctg]
        aln.reference_start = hit.r_st
        aln.mapping_quality = int(hit.mapq)
        aln.cigarstring = full_cigar(hit, len(seq))
        if hit.strand < 0:
            aln.query_sequence = revcomp(seq)
            aln.query_qualities = pysam.qualitystring_to_array(qual[::-1])
        else:
            aln.query_sequence = seq
            aln.query_qualities = pysam.qualitystring_to_array(qual)
        aln.set_tag('NM', int(hit.NM), value_type='i')
        bam.write(aln)
        written += 1
print('written alignments', written)

pysam.sort('-o', str(SORTED_BAM), str(UNSORTED_BAM))
pysam.index(str(SORTED_BAM))
print('sorted bam', SORTED_BAM)

variant_lookup = VariantLookup(None)
counts = {'train': 0, 'val': 0, 'test': 0}
files = {
    'train': open(TRAIN_JSONL, 'w', encoding='utf-8'),
    'val': open(VAL_JSONL, 'w', encoding='utf-8'),
    'test': open(TEST_JSONL, 'w', encoding='utf-8'),
}
try:
    with pysam.AlignmentFile(str(SORTED_BAM), 'rb', reference_filename=str(REFERENCE_FA)) as bam:
        for aln in bam.fetch(until_eof=True):
            enc = build_read_encoding(aln, fasta, max_insertions_per_pos=MAX_INSERTIONS_PER_POS)
            if enc is None or len(enc.target_bases) < 512:
                continue
            digest = int(hashlib.md5(enc.read_id.encode()).hexdigest(), 16) % 10
            split = 'train' if digest < 8 else ('val' if digest == 8 else 'test')
            for ws, we in iter_windows(len(enc.target_bases), WINDOW_SIZE, WINDOW_OVERLAP, 256):
                build_window_kwargs = dict(
                    target_aln=aln,
                    read_encoding=enc,
                    bam=bam,
                    variant_lookup=variant_lookup,
                    confidence_lookup=None,
                    window_start=ws,
                    window_end=we,
                    max_supports=MAX_SUPPORTS,
                    min_mapq=MIN_MAPQ,
                    min_confident_fraction=0.0,
                    min_mapped_fraction=0.6,
                    support_disagreement_threshold=0.7,
                    min_support_depth=2,
                    max_insertions_per_pos=MAX_INSERTIONS_PER_POS,
                )
                if 'min_supports_per_window' in inspect.signature(build_window_example).parameters:
                    build_window_kwargs['min_supports_per_window'] = MIN_SUPPORTS_PER_WINDOW
                ex = build_window_example(**build_window_kwargs)
                if ex is None:
                    continue
                files[split].write(json.dumps(ex) + '\n')
                counts[split] += 1
finally:
    for handle in files.values():
        handle.close()
    fasta.close()

manifest = {
    'selected_reads': len(reads),
    'written_alignments': written,
    'window_counts': counts,
    'train_path': str(TRAIN_JSONL),
    'val_path': str(VAL_JSONL),
    'test_path': str(TEST_JSONL),
    'bam_path': str(SORTED_BAM),
}
MANIFEST.write_text(json.dumps(manifest, indent=2))
manifest


selected reads 1000
written alignments 434
sorted bam /Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/aligned.sorted.bam


{'selected_reads': 1000,
 'written_alignments': 434,
 'window_counts': {'train': 1152, 'val': 174, 'test': 127},
 'train_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/train.jsonl',
 'val_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/val.jsonl',
 'test_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/test.jsonl',
 'bam_path': '/Users/shanejayasundera/LRS-Error-Correction/data/real_uti89_notebook/aligned.sorted.bam'}

In [85]:
from collections import Counter
import json
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/shanejayasundera/LRS-Error-Correction')
RUN_DIR = ROOT / 'data' / 'real_uti89_notebook'
TRAIN_JSONL = RUN_DIR / 'train.jsonl'
VAL_JSONL = RUN_DIR / 'val.jsonl'
TEST_JSONL = RUN_DIR / 'test.jsonl'

ID_TO_EDIT = {
    0: 'COPY',
    1: 'SUB_A',
    2: 'SUB_C',
    3: 'SUB_G',
    4: 'SUB_T',
    5: 'DEL',
    6: 'INS_A',
    7: 'INS_C',
    8: 'INS_G',
    9: 'INS_T',
    10: 'PAD',
}


def count_edit_labels(path: Path):
    all_tokens = Counter()
    core_tokens = Counter()
    insertion_tokens = Counter()
    windows = 0
    positions = 0

    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            item = json.loads(line)
            windows += 1
            for label_slots in item['edit_labels']:
                positions += 1
                for slot_idx, token_id in enumerate(label_slots):
                    token = ID_TO_EDIT.get(token_id, str(token_id))
                    all_tokens[token] += 1
                    if slot_idx == len(label_slots) - 1:
                        core_tokens[token] += 1
                    elif token != 'PAD':
                        insertion_tokens[token] += 1

    def to_df(counter: Counter, split: str) -> pd.DataFrame:
        total = sum(counter.values()) or 1
        rows = [
            {
                'split': split,
                'token': token,
                'count': count,
                'fraction': count / total,
            }
            for token, count in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
        ]
        return pd.DataFrame(rows)

    summary = pd.DataFrame([
        {
            'split': path.stem,
            'windows': windows,
            'positions': positions,
            'all_token_total': sum(all_tokens.values()),
            'core_token_total': sum(core_tokens.values()),
            'non_pad_insertion_total': sum(insertion_tokens.values()),
        }
    ])
    return summary, to_df(all_tokens, path.stem), to_df(core_tokens, path.stem), to_df(insertion_tokens, path.stem)


split_paths = [TRAIN_JSONL, VAL_JSONL, TEST_JSONL]
summary_frames = []
all_frames = []
core_frames = []
ins_frames = []

for split_path in split_paths:
    if not split_path.exists():
        raise FileNotFoundError(f'Missing dataset split: {split_path}')
    summary_df, all_df, core_df, ins_df = count_edit_labels(split_path)
    summary_frames.append(summary_df)
    all_frames.append(all_df)
    core_frames.append(core_df)
    ins_frames.append(ins_df)

label_summary_df = pd.concat(summary_frames, ignore_index=True)
all_label_counts_df = pd.concat(all_frames, ignore_index=True)
core_label_counts_df = pd.concat(core_frames, ignore_index=True)
insertion_label_counts_df = pd.concat(ins_frames, ignore_index=True)

print('Dataset summary')
print(label_summary_df.to_string(index=False))
print()
print('Core-slot label counts')
print(core_label_counts_df.to_string(index=False))
print()
print('Insertion-slot label counts (PAD removed)')
print(insertion_label_counts_df.to_string(index=False))


Dataset summary
split  windows  positions  all_token_total  core_token_total  non_pad_insertion_total
train     1152    1116868          3350604           1116868                   144740
  val      174     171031           513093            171031                    21520
 test      127     125167           375501            125167                    15855

Core-slot label counts
split token  count  fraction
train  COPY 884647  0.792078
train   DEL  73799  0.066077
train SUB_C  44046  0.039437
train SUB_G  40693  0.036435
train SUB_T  36999  0.033127
train SUB_A  36684  0.032845
  val  COPY 134597  0.786974
  val   DEL  12084  0.070654
  val SUB_C   6615  0.038677
  val SUB_G   6325  0.036982
  val SUB_T   5833  0.034105
  val SUB_A   5577  0.032608
 test  COPY  99179  0.792373
 test   DEL   8417  0.067246
 test SUB_C   4714  0.037662
 test SUB_G   4704  0.037582
 test SUB_A   4577  0.036567
 test SUB_T   3576  0.028570

Insertion-slot label counts (PAD removed)
split token  count  fr

In [86]:
ALL_RUNS = [
    {
        'name': 'full',
        'support_mode': 'full',
        'description': 'Target + overlap-conditioned support encoder',
    },
    {
        'name': 'target_only',
        'support_mode': 'target_only',
        'description': 'Target encoder only; support path disabled',
    },
    {
        'name': 'masked_target',
        'support_mode': 'masked_target',
        'description': 'Support-conditioned with masked target features',
    },
]

SELECTED_RUNS = ['full']
selected_run_set = set(SELECTED_RUNS)
unknown_runs = sorted(selected_run_set - {run['name'] for run in ALL_RUNS})
if unknown_runs:
    raise ValueError(f'Unknown run names requested: {unknown_runs}')

ABLATION_RUNS = [run for run in ALL_RUNS if run['name'] in selected_run_set]
if not ABLATION_RUNS:
    raise ValueError('SELECTED_RUNS is empty; choose at least one of full, target_only, masked_target.')

ABLATION_CONFIG_DIR = ROOT / 'configs' / 'real_uti89_notebook_ablation'
ABLATION_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

base_config = {
    'model': {
        'd_model': 32,
        'num_heads': 4,
        'num_layers': 1,
        'ff_mult': 2,
        'dropout': 0.1,
        'support_feature_dim': 12,
        'max_supports': 8,
        'max_insertions_per_pos': 2,
        'conv_kernel_size': 5,
        'support_mode': 'full',
        'apply_hard_edit_support_filter': True,
        'hard_edit_min_support_agreement': 0.95,
        'hard_edit_max_support_entropy': 0.15,
        'hard_edit_min_support_depth': 2.0,
        'hard_edit_filter_logit_bias': 6.0,
        'inference_hard_edit_confidence_threshold': 0.9,
        'inference_hard_edit_temperature': 0.75,
    },
    'data': {
        'train_path': str(TRAIN_JSONL.relative_to(ROOT)),
        'val_path': str(VAL_JSONL.relative_to(ROOT)),
        'test_path': str(TEST_JSONL.relative_to(ROOT)),
        'max_target_len': 1024,
        'max_supports': 8,
        'max_support_len': 1024,
        'num_workers': 0,
    },
    'train': {
        'seed': 42,
        'batch_size': 4,
        'epochs': 5,
        'lr': 1e-4,
        'weight_decay': 0.01,
        'grad_clip_norm': 0.5,
        'device': 'cpu',
        'mixed_precision': False,
        'log_every': 20,
        'eval_every': 1,
        'save_dir': '',
        'checkpoint_metric': 'composite_sequence',
        'checkpoint_metric_mode': 'max',
        'checkpoint_overcorrection_weight': 3.0,
        'checkpoint_length_ratio_weight': 0.25,
    },
    'loss': {
        'lambda_edit': 1.0,
        'lambda_sequence': 0.5,
        'lambda_length': 0.3,
        'lambda_insertion_count': 0.2,
        'lambda_trust': 0.1,
        'lambda_hard_edit': 0.4,
        'lambda_hard_edit_precision': 0.8,
        'lambda_selective_hard_edit': 0.2,
        'lambda_support': 0.0,
        'lambda_preserve': 0.15,
        'lambda_uncertainty': 0.1,
        'homopolymer_weight_scale': 0.15,
        'label_smoothing': 0.0,
        'edit_class_weights': [],
        'auto_edit_class_weights': True,
        'edit_class_weight_power': 1.0,
        'edit_class_weight_count_smoothing': 1.0,
        'edit_class_weight_min': 0.25,
        'edit_class_weight_max': 8.0,
        'substitution_loss_scale': 1.75,
        'deletion_loss_scale': 1.25,
        'insertion_loss_scale': 1.1,
        'hard_edit_uncertainty_power': 2.0,
        'hard_edit_entropy_threshold': 0.15,
        'hard_edit_agreement_threshold': 0.95,
        'hard_edit_entropy_scale': 2.5,
        'hard_edit_low_agreement_scale': 3.5,
        'hard_edit_false_positive_weight': 8.0,
        'hard_edit_false_negative_weight': 1.0,
        'selective_hard_edit_confidence_threshold': 0.6,
        'selective_hard_edit_uncertainty_threshold': 0.4,
        'selective_hard_edit_min_support_agreement': 0.93,
    },
}

for run in ABLATION_RUNS:
    run_name = run['name']
    cfg = copy.deepcopy(base_config)
    cfg['model']['support_mode'] = run['support_mode']
    cfg['train']['save_dir'] = str((ROOT / 'checkpoints' / 'real_uti89_notebook_ablation' / run_name).relative_to(ROOT))
    config_path = ABLATION_CONFIG_DIR / f'{run_name}.yaml'
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    run['config_path'] = config_path
    run['checkpoint_dir'] = ROOT / cfg['train']['save_dir']
    run['output_dir'] = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / run_name
    run['output_dir'].mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([
    {
        'run': run['name'],
        'support_mode': run['support_mode'],
        'config_path': str(run['config_path'].relative_to(ROOT)),
        'checkpoint_dir': str(run['checkpoint_dir'].relative_to(ROOT)),
        'output_dir': str(run['output_dir'].relative_to(ROOT)),
    }
    for run in ABLATION_RUNS
]))


,run,support_mode,config_path,checkpoint_dir,output_dir
0,full,full,configs/real_uti89_notebook_ablation/full.yaml,checkpoints/real_uti89_notebook_ablation/full,outputs/real_uti89_notebook_ablation/full


In [87]:
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src')
env['PYTHONUNBUFFERED'] = '1'

for run in ABLATION_RUNS:
    train_cmd = [
        sys.executable,
        'scripts/train.py',
        '--config',
        str(run['config_path']),
    ]
    print(f"\n=== training {run['name']} ({run['description']}) ===")
    print('running', ' '.join(train_cmd))
    subprocess.run(train_cmd, cwd=ROOT, env=env, check=True)



=== training full (Target + overlap-conditioned support encoder) ===
running /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/train.py --config /Users/shanejayasundera/LRS-Error-Correction/configs/real_uti89_notebook_ablation/full.yaml


/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:232: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.mixed_precision and device.type == "cuda")


resolved edit class weights:
[
  {
    "token_id": 0,
    "token": "COPY",
    "weight": 0.25
  },
  {
    "token_id": 1,
    "token": "SUB_A",
    "weight": 3.439107656478882
  },
  {
    "token_id": 2,
    "token": "SUB_C",
    "weight": 2.8642964363098145
  },
  {
    "token_id": 3,
    "token": "SUB_G",
    "weight": 3.1003012657165527
  },
  {
    "token_id": 4,
    "token": "SUB_T",
    "weight": 3.4098286628723145
  },
  {
    "token_id": 5,
    "token": "DEL",
    "weight": 1.709534764289856
  },
  {
    "token_id": 6,
    "token": "INS_A",
    "weight": 3.622270107269287
  },
  {
    "token_id": 7,
    "token": "INS_C",
    "weight": 3.3582746982574463
  },
  {
    "token_id": 8,
    "token": "INS_G",
    "weight": 3.305222988128662
  },
  {
    "token_id": 9,
    "token": "INS_T",
    "weight": 3.691694736480713
  },
  {
    "token_id": 10,
    "token": "PAD",
    "weight": 0.0
  }
]
epoch 1/5


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision):


train step=20: {"loss": 13.347130489349365, "edit_loss": 8.300959539413451, "sequence_loss": 1.4190350770950317, "length_loss": 0.19078357685357333, "insertion_count_loss": 0.7056259900331497, "trust_loss": 0.7024859935045242, "hard_edit_loss": 0.0010430453708977438, "hard_edit_precision_loss": 4.0687971949577335, "selective_hard_edit_loss": 2.8908380389213564, "hard_edit_entropy_excess_mean": 0.04970773672685027, "hard_edit_agreement_deficit_mean": 0.2599911291152239, "support_loss": 3.865794229507446, "preserve_loss": 0.7656679153442383, "uncertainty_loss": 1.195717591047287, "predicted_length": 994.2262023925781, "target_length": 1024.725, "predicted_insertions": 787.7159362792969, "target_insertions": 123.375, "trust_gate_mean": 0.49526817798614503, "support_confidence_mean": 0.5203835248947144, "support_uncertainty_mean": 0.27359371464699506, "selective_hard_edit_fraction": 0.03360595703125, "edit_accuracy": 0.07548655569553375, "core_edit_accuracy": 0.0678749920334667, "copy_reca

/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision):


{
  "train_loss": 11.903698159588707,
  "train_edit_loss": 7.193187089429961,
  "train_sequence_loss": 1.3682625964283943,
  "train_length_loss": 0.251920798217826,
  "train_insertion_count_loss": 0.7546895585126348,
  "train_trust_loss": 0.7040774018193284,
  "train_hard_edit_loss": 0.005085093848669607,
  "train_hard_edit_precision_loss": 3.726835550947322,
  "train_selective_hard_edit_loss": 2.98425406921241,
  "train_hard_edit_entropy_excess_mean": 0.03510022216242861,
  "train_hard_edit_agreement_deficit_mean": 0.24644633021671325,
  "train_support_loss": 3.1451068934467106,
  "train_preserve_loss": 0.5159763500301374,
  "train_uncertainty_loss": 0.7170806809638938,
  "train_predicted_length": 1223.0241947174072,
  "train_target_length": 1031.084201388889,
  "train_predicted_insertions": 867.3221465216743,
  "train_target_insertions": 125.64236111111111,
  "train_trust_gate_mean": 0.6366862445655797,
  "train_support_confidence_mean": 0.5399870638632112,
  "train_support_uncertain

In [88]:
for run in ABLATION_RUNS:
    test_summary = run['output_dir'] / 'test_summary.json'
    test_predictions = run['output_dir'] / 'test_predictions.jsonl'
    test_cmd = [
        sys.executable,
        'scripts/test.py',
        '--config',
        str(run['config_path']),
        '--checkpoint',
        str(run['checkpoint_dir'] / 'best_model.pt'),
        '--summary-out',
        str(test_summary),
        '--predictions-out',
        str(test_predictions),
    ]
    print(f"\n=== testing {run['name']} ===")
    print('running', ' '.join(test_cmd))
    subprocess.run(test_cmd, cwd=ROOT, env=env, check=True)
    run['test_summary_path'] = test_summary
    run['test_predictions_path'] = test_predictions



=== testing full ===
running /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/test.py --config /Users/shanejayasundera/LRS-Error-Correction/configs/real_uti89_notebook_ablation/full.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/checkpoints/real_uti89_notebook_ablation/full/best_model.pt --summary-out /Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/full/test_summary.json --predictions-out /Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/full/test_predictions.jsonl


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/shanejayasundera/LRS-Error-Correction/scripts/test.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


{
  "loss": 6.429938703775406,
  "edit_loss": 3.527422621846199,
  "sequence_loss": 1.0736896097660065,
  "length_loss": 0.42544399481266737,
  "insertion_count_loss": 0.4850489627569914,
  "trust_loss": 0.759575198404491,
  "hard_edit_loss": 0.04957885731710121,
  "hard_edit_precision_loss": 2.2158421352505684,
  "selective_hard_edit_loss": 1.0372051587328315,
  "hard_edit_entropy_excess_mean": 0.06497689852767508,
  "hard_edit_agreement_deficit_mean": 0.24517937982454896,
  "support_loss": 1.364142656326294,
  "preserve_loss": 0.11089748283848166,
  "uncertainty_loss": 0.484898102004081,
  "predicted_length": 1477.3508987426758,
  "target_length": 1044.0208320617676,
  "predicted_insertions": 623.4266090393066,
  "target_insertions": 124.69791674613953,
  "trust_gate_mean": 0.8418195210397243,
  "support_confidence_mean": 0.6281506596133113,
  "support_uncertainty_mean": 0.22953304997645319,
  "selective_hard_edit_fraction": 0.06103769934270531,
  "edit_accuracy": 0.7056688833981752,

In [89]:
comparison_rows = []
history_frames = []

for run in ABLATION_RUNS:
    history_path = run['checkpoint_dir'] / 'history.json'
    test_summary_path = run['test_summary_path']
    history = json.loads(history_path.read_text())
    test_metrics = json.loads(test_summary_path.read_text())
    history_df = pd.DataFrame(history)
    history_df.insert(0, 'run', run['name'])
    history_frames.append(history_df)

    comparison_rows.append({
        'run': run['name'],
        'support_mode': run['support_mode'],
        'description': run['description'],
        'best_epoch': int(history_df.iloc[history_df['val_selection_score'].idxmax()]['epoch']),
        'val_selection_score_max': float(history_df['val_selection_score'].max()),
        'final_val_sequence_identity': float(history_df.iloc[-1]['val_sequence_identity']),
        'final_val_length_ratio': float(history_df.iloc[-1]['val_predicted_length_ratio']),
        'final_val_overcorrection_rate': float(history_df.iloc[-1]['val_overcorrection_rate']),
        'final_val_sub_precision': float(history_df.iloc[-1]['val_sub_precision']),
        'final_val_delete_precision': float(history_df.iloc[-1]['val_delete_precision']),
        'final_val_high_support_agreement_sequence_identity': float(history_df.iloc[-1]['val_high_support_agreement_sequence_identity']),
        'final_val_low_support_agreement_sequence_identity': float(history_df.iloc[-1]['val_low_support_agreement_sequence_identity']),
        'test_sequence_identity': float(test_metrics['sequence_identity']),
        'test_sequence_normalized_edit_distance': float(test_metrics['sequence_normalized_edit_distance']),
        'test_predicted_length_ratio': float(test_metrics['predicted_length_ratio']),
        'test_predicted_insertions_per_window': float(test_metrics['predicted_insertions_per_window']),
        'test_predicted_insertion_burst_length': float(test_metrics['predicted_insertion_burst_length']),
        'test_sub_precision': float(test_metrics['sub_precision']),
        'test_sub_recall': float(test_metrics['sub_recall']),
        'test_delete_precision': float(test_metrics['delete_precision']),
        'test_delete_recall': float(test_metrics['delete_recall']),
        'test_high_support_agreement_sequence_identity': float(test_metrics['high_support_agreement_sequence_identity']),
        'test_low_support_agreement_sequence_identity': float(test_metrics['low_support_agreement_sequence_identity']),
        'test_overcorrection_rate': float(test_metrics['overcorrection_rate']),
    })

ablation_history_df = pd.concat(history_frames, ignore_index=True)
ablation_summary_df = pd.DataFrame(comparison_rows).sort_values(
    by=['test_sequence_identity', 'test_sequence_normalized_edit_distance'],
    ascending=[False, True],
).reset_index(drop=True)

ablation_summary_path = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / 'ablation_summary.json'
ablation_history_path = ROOT / 'outputs' / 'real_uti89_notebook_ablation' / 'ablation_history.json'
with open(ablation_summary_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_rows, f, indent=2)
with open(ablation_history_path, 'w', encoding='utf-8') as f:
    json.dump(ablation_history_df.to_dict(orient='records'), f, indent=2)

print(ablation_summary_path)
print(ablation_history_path)
display(ablation_summary_df)
display(ablation_history_df)


/Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/ablation_summary.json
/Users/shanejayasundera/LRS-Error-Correction/outputs/real_uti89_notebook_ablation/ablation_history.json


,run,support_mode,description,best_epoch,val_selection_score_max,final_val_sequence_identity,final_val_length_ratio,final_val_overcorrection_rate,final_val_sub_precision,final_val_delete_precision,...,test_predicted_length_ratio,test_predicted_insertions_per_window,test_predicted_insertion_burst_length,test_sub_precision,test_sub_recall,test_delete_precision,test_delete_recall,test_high_support_agreement_sequence_identity,test_low_support_agreement_sequence_identity,test_overcorrection_rate
0,full,full,Target + overlap-conditioned support encoder,5,0.671431,0.676477,0.979814,0.0,0.390358,0.0,...,0.974087,30.052083,1.052542,0.638591,0.007885,0.0,0.0,0.42307,0.231269,0.0


,run,train_loss,train_edit_loss,train_sequence_loss,train_length_loss,train_insertion_count_loss,train_trust_loss,train_hard_edit_loss,train_hard_edit_precision_loss,train_selective_hard_edit_loss,...,val_sub_pred_count_high_entropy,val_sub_pred_count_low_agreement,val_sub_pred_count_high_agreement,val_delete_pred_count_low_entropy,val_delete_pred_count_high_entropy,val_delete_pred_count_low_agreement,val_delete_pred_count_high_agreement,val_overcorrection_rate,epoch,val_selection_score
0,full,11.903698,7.193187,1.368263,0.251921,0.754690,0.704077,0.005085,3.726836,2.984254,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,1,0.638190
1,full,9.758267,5.224327,1.263183,0.338097,0.604214,0.866769,0.029053,3.864636,2.030675,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,2,0.665234
2,full,8.504898,4.598285,1.206831,0.379575,0.547576,0.998282,0.040898,3.218752,1.593994,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,3,0.635050
3,full,7.282701,4.031704,1.142022,0.407466,0.508652,1.068279,0.047893,2.491023,1.391908,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.00003,4,0.662032
4,full,6.863817,3.755798,1.115208,0.406364,0.498324,0.988530,0.054253,2.417320,1.081052,...,0.0,0.0,2.318182,0.0,0.0,0.0,0.0,0.00000,5,0.671431
